# Chapter 3 — Description Logics
### Notebook 0 · Overview and setup

*Book reference: Keet, *Ontology Engineering* (2nd ed.), Ch. 3*

Chapter 2 ended at a wall: first-order logic is undecidable, so no terminating procedure can be complete for it. Description logics are the engineering response — **fragments of FOL chosen so that reasoning terminates**. In this chapter you build the reasoner that makes that claim true.

In [1]:
import sys, os, json, textwrap
from pathlib import Path

# Make the repo root importable no matter where Jupyter was started.
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate))
        break

import oe_course
print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


In [2]:
import sys, json, logging
from pathlib import Path
here = Path.cwd()
for candidate in [here, *here.parents]:
    if (candidate / "oe_course").is_dir():
        sys.path.insert(0, str(candidate)); break
sys.path.insert(0, str(Path.cwd()))

import ch03_toolkit as dl
import pandas as pd
A = dl.Atomic
logging.getLogger("dspy").setLevel(logging.WARNING)

In [3]:
import oe_course; print(json.dumps(oe_course.describe_environment(), indent=1))

{
 "mode": "offline (simulated LLM)",
 "chat_model": "claude-opus-5",
 "dspy_model": "anthropic/claude-opus-5",
 "fuseki": "in-memory rdflib",
 "artifacts": "C:\\Users\\marci\\OneDrive\\DEV\\EDU\\AIML\\Graph ML\\Ontology Engineering\\course\\artifacts"
}


## Notebooks in this chapter

| # | Notebook | Book section | What you build |
|---|---|---|---|
| 0 | `00_overview_and_setup` | — | a working DL environment |
| 1 | `01_dl_basics_and_tableau` | 3.1 | **an ALC tableau reasoner**, with blocking |
| 2 | `02_important_dls` | 3.2 | a DL namer, and the measured cost of expressivity |
| 3 | `03_reasoning_services` | 3.3 | satisfiability, subsumption, classification |
| 4 | `04_exercises` | 3.4 | autograded answers |
| 5 | `05_agentic_lab` | — | an agent that names logics and *pays* for soundness |


**By the end of this notebook you can:**

1. Read and write DL concept expressions, and say what each constructor costs you.
2. Run — and explain — a **tableau**: why `∃r.A ⊓ ∀r.¬A` is unsatisfiable, step by step.
3. Explain why **blocking** is what makes cyclic axioms terminate.
4. Name the DL a knowledge base actually needs, and predict the reasoning complexity you just signed up for.
5. Derive every reasoning service from concept satisfiability.
6. Build an agent that decides *when a sound reasoner is worth its cost*.

## The engine

`ch03_toolkit` is a self-contained DL implementation: concept AST, negation normal form, expressivity analysis, and a tableau reasoner for **ALC** with TBox internalisation and subset blocking. No external reasoner, no Java.

> **Scope, stated up front.** The tableau decides ALC. Number restrictions, inverses and transitive roles are *analysed* by the DL namer but **not reasoned over** — implementing SHIQ is a research-grade exercise, not a notebook. Every place this matters, the notebooks say so.

### Sanity check: the smallest interesting unsatisfiability

In [4]:
c = dl.And(dl.Exists('r', A('A')), dl.ForAll('r', dl.Not(A('A'))))
print('concept   :', dl.to_string(c))
result = dl.satisfiable(c)
print('verdict   :', result.summary())
assert not result.satisfiable
print('\nIt demands an r-successor that is an A, while insisting every\n'
      'r-successor is not an A. No model can satisfy both.')

concept   : (exists r.A and forall r.not A)
verdict   : UNSATISFIABLE after 4 rule applications, 0 branch points, max depth 1

It demands an r-successor that is an A, while insisting every
r-successor is not an A. No model can satisfy both.
